# Modelos

> Modelo base : Naive estacional

> Modelo experimentos : SARIMA

In [7]:
import pandas as pd
from sklearn.metrics import mean_absolute_percentage_error as mape, root_mean_squared_error as rmse
from statsforecast.models import ARIMA
from statsforecast import StatsForecast

import sys
sys.path.append('..')

from utils.estacionariedad import prueba_diferenciacion
from utils.parametros_ARIMA import pacf_acf
from utils.metricas_cv import metricas_cv

In [8]:
df = pd.read_csv('../datos/df_01.csv',
                parse_dates = ['fecha'],
                index_col = 'fecha')
df.head()

,value
fecha,
2019-01-01 00:00:00,23216.0
2019-01-01 01:00:00,24947.0
2019-01-01 02:00:00,27655.0
2019-01-01 03:00:00,27676.0
2019-01-01 04:00:00,26853.0


### Preparar formato 

In [9]:
df = df.reset_index()                   
df.columns = ['ds', 'y']                      
df['unique_id'] = 'serie1'
df = df[['unique_id', 'ds', 'y']]
df

,unique_id,ds,y
0,serie1,2019-01-01 00:00:00,23216.0
1,serie1,2019-01-01 01:00:00,24947.0
2,serie1,2019-01-01 02:00:00,27655.0
3,serie1,2019-01-01 03:00:00,27676.0
4,serie1,2019-01-01 04:00:00,26853.0
...,...,...,...
57404,serie1,2025-07-19 20:00:00,29380.0
57405,serie1,2025-07-19 21:00:00,27763.0
57406,serie1,2025-07-19 22:00:00,26659.0
57407,serie1,2025-07-19 23:00:00,26438.0


### Modelo Seasonal Naive

In [10]:
H = 24
df_train_naive = df.iloc[:-H]['y'].copy()
df_test_naive = df.iloc[-H]['y'].copy()

df_pred_naive = df_train_naive.shift(H)

mape_base = mape(df_train_naive[H:], df_pred_naive[H:])
rmse_base = rmse(df_train_naive[H:], df_pred_naive[H:])

print('> Medidas desempeño --- Seasonal Naive')
print(f'MAPE: {100*mape_base:.4f}')
print(f'RMSE: {rmse_base:.4f}')

> Medidas desempeño --- Seasonal Naive
MAPE: 5.0687
RMSE: 1821.9338


### Modelo SARIMA
* Una sola variable
* Tendencia y varianza constantes
* Serie estacionaria (?)

### Selección del mejor modelo SARIMA

Basándome en el comportamiento del ACF, PACF obtenidos en el EDA:

| Parámetro | Valor | Fuente | Razón |
|-----------|-------|--------|-------|
| p | 2 | PACF regular | Corte brusco tras lag 2 |
| d | 0 | Dickey-Fuller (sin diferencia) | Serie ya estacionaria |
| q | 0 | ACF regular | No hay corte brusco |
| P | 1 | PACF estacional (lag 24) | Un pico significativo en lag 24 |
| D | 1 | ACF estacional | Picos en 24/48/72 decaen lento |
| Q | 0 | ACF estacional | No hay corte brusco en lags estacionales |
| s | 24 | Ambas (ACF/PACF) | Picos en 24, 48, 72... |

In [ ]:
ps, d, qs = [0, 1, 2], 1, [0, 1, 2]
P, D, Qs = 0, 1, [0, 1, 2]

modelos = [ARIMA(order=(p, d, q), seasonal_order=(P, D, Q), season_length=24, alias=f'ARIMA({p},{d},{q})({P},{D},{Q})_24', include_constant=False) for p in ps for q in qs for Q in Qs] 

sf = StatsForecast(
    models = modelos,
    freq = 'h',
    n_jobs = -1
)

print(f'Se crearon {len(modelos)} modelos')
modelos

Se crearon 27 modelos


[ARIMA(0,1,0)(0,1,0)_24,
 ARIMA(0,1,0)(0,1,1)_24,
 ARIMA(0,1,0)(0,1,2)_24,
 ARIMA(0,1,1)(0,1,0)_24,
 ARIMA(0,1,1)(0,1,1)_24,
 ARIMA(0,1,1)(0,1,2)_24,
 ARIMA(0,1,2)(0,1,0)_24,
 ARIMA(0,1,2)(0,1,1)_24,
 ARIMA(0,1,2)(0,1,2)_24,
 ARIMA(1,1,0)(0,1,0)_24,
 ARIMA(1,1,0)(0,1,1)_24,
 ARIMA(1,1,0)(0,1,2)_24,
 ARIMA(1,1,1)(0,1,0)_24,
 ARIMA(1,1,1)(0,1,1)_24,
 ARIMA(1,1,1)(0,1,2)_24,
 ARIMA(1,1,2)(0,1,0)_24,
 ARIMA(1,1,2)(0,1,1)_24,
 ARIMA(1,1,2)(0,1,2)_24,
 ARIMA(2,1,0)(0,1,0)_24,
 ARIMA(2,1,0)(0,1,1)_24,
 ARIMA(2,1,0)(0,1,2)_24,
 ARIMA(2,1,1)(0,1,0)_24,
 ARIMA(2,1,1)(0,1,1)_24,
 ARIMA(2,1,1)(0,1,2)_24,
 ARIMA(2,1,2)(0,1,0)_24,
 ARIMA(2,1,2)(0,1,1)_24,
 ARIMA(2,1,2)(0,1,2)_24]

In [109]:
ps, d, qs = [0, 1, 2], 0, [0, 1, 2]
P, D, Qs = 0, 0, [0, 1, 2]

modelos = [ARIMA(order=(p, d, q), seasonal_order=(P, D, Q), season_length=24, alias=f'ARIMA({p},{d},{q})({P},{D},{Q})_24', include_constant=False) for p in ps for q in qs for Q in Qs] 

sf = StatsForecast(
    models = modelos,
    freq = 'h',
    n_jobs = -1
)

print(f'Se crearon {len(modelos)} modelos')
modelos

Se crearon 27 modelos


[ARIMA(0,0,0)(0,0,0)_24,
 ARIMA(0,0,0)(0,0,1)_24,
 ARIMA(0,0,0)(0,0,2)_24,
 ARIMA(0,0,1)(0,0,0)_24,
 ARIMA(0,0,1)(0,0,1)_24,
 ARIMA(0,0,1)(0,0,2)_24,
 ARIMA(0,0,2)(0,0,0)_24,
 ARIMA(0,0,2)(0,0,1)_24,
 ARIMA(0,0,2)(0,0,2)_24,
 ARIMA(1,0,0)(0,0,0)_24,
 ARIMA(1,0,0)(0,0,1)_24,
 ARIMA(1,0,0)(0,0,2)_24,
 ARIMA(1,0,1)(0,0,0)_24,
 ARIMA(1,0,1)(0,0,1)_24,
 ARIMA(1,0,1)(0,0,2)_24,
 ARIMA(1,0,2)(0,0,0)_24,
 ARIMA(1,0,2)(0,0,1)_24,
 ARIMA(1,0,2)(0,0,2)_24,
 ARIMA(2,0,0)(0,0,0)_24,
 ARIMA(2,0,0)(0,0,1)_24,
 ARIMA(2,0,0)(0,0,2)_24,
 ARIMA(2,0,1)(0,0,0)_24,
 ARIMA(2,0,1)(0,0,1)_24,
 ARIMA(2,0,1)(0,0,2)_24,
 ARIMA(2,0,2)(0,0,0)_24,
 ARIMA(2,0,2)(0,0,1)_24,
 ARIMA(2,0,2)(0,0,2)_24]

In [110]:
H = 24
df_train = df.iloc[:-H].copy()
df_test = df.iloc[-H].copy()

### Validación cruzada

In [111]:
cv_rolling = sf.cross_validation(
    df = df_train,
    h = 24, 
    step_size = 48,
    n_windows = 15,
    input_size = 168 # entrena 7 días
    )

In [103]:
cv_rolling

,unique_id,ds,cutoff,y,"ARIMA(0,1,0)(0,1,0)_24","ARIMA(0,1,0)(0,1,1)_24","ARIMA(0,1,0)(0,1,2)_24","ARIMA(0,1,1)(0,1,0)_24","ARIMA(0,1,1)(0,1,1)_24","ARIMA(0,1,1)(0,1,2)_24",...,"ARIMA(1,1,2)(0,1,2)_24","ARIMA(2,1,0)(0,1,0)_24","ARIMA(2,1,0)(0,1,1)_24","ARIMA(2,1,0)(0,1,2)_24","ARIMA(2,1,1)(0,1,0)_24","ARIMA(2,1,1)(0,1,1)_24","ARIMA(2,1,1)(0,1,2)_24","ARIMA(2,1,2)(0,1,0)_24","ARIMA(2,1,2)(0,1,1)_24","ARIMA(2,1,2)(0,1,2)_24"
0,serie1,2025-06-20 01:00:00,2025-06-20,29138.0,30203.0,29986.212109,30138.269638,29780.102040,29623.218588,29771.108750,...,29698.173756,29704.369794,29516.191102,29692.934475,29961.191455,29517.363180,29695.957580,29701.440048,29637.676462,29806.775345
1,serie1,2025-06-20 02:00:00,2025-06-20,30352.0,31858.0,31590.550990,31632.619405,31435.102040,31218.481851,31293.709531,...,31055.090179,31174.619515,30893.745474,31042.979650,31217.259129,30892.694085,31046.629921,31166.178550,31184.700621,31279.568401
2,serie1,2025-06-20 03:00:00,2025-06-20,31662.0,32483.0,32417.381391,32576.805314,32060.102040,32072.529411,32215.987223,...,31876.001957,31731.326224,31636.094895,31884.531998,31887.673252,31624.771475,31883.525964,31739.932971,32034.854561,32210.849221
3,serie1,2025-06-20 04:00:00,2025-06-20,31344.0,32383.0,32390.830265,32700.017586,31960.102040,32063.033421,32311.548365,...,31912.826063,31606.084530,31572.507460,31943.519808,31622.750227,31549.838808,31936.354893,31625.333225,32009.386679,32312.997068
4,serie1,2025-06-20 05:00:00,2025-06-20,31020.0,31377.0,31491.943431,31869.390152,30954.102040,31180.665014,31478.787076,...,31042.905350,30590.755077,30662.103889,31090.105100,30689.984901,30630.232669,31077.510184,30615.216551,31129.443721,31498.361186
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,serie1,2025-07-18 20:00:00,2025-07-18,31447.0,31519.0,30812.566467,30745.285436,31327.025113,30353.790248,30138.681565,...,30055.235911,31252.669505,30367.419139,30171.465272,31241.548954,30364.139441,30157.562649,31571.114407,30397.090160,30185.607049
356,serie1,2025-07-18 21:00:00,2025-07-18,31102.0,31198.0,30526.874960,30475.037068,31006.025113,30064.733518,29897.509910,...,29810.417215,30931.669473,30081.589037,29924.479822,30920.548852,30078.254948,29911.878333,31247.815282,30111.169751,29938.927956
357,serie1,2025-07-18 22:00:00,2025-07-18,31182.0,30920.0,30459.467502,30441.359686,30728.025113,30011.393968,29941.905293,...,29843.577430,30653.669458,30015.042026,29932.741100,30642.548802,30012.016738,29923.172957,30969.628129,30045.094789,29949.421535
358,serie1,2025-07-18 23:00:00,2025-07-18,30418.0,31569.0,30810.421814,30745.348822,31377.025113,30340.168229,30111.579447,...,30033.812178,31302.669451,30364.678080,30168.746078,31291.548778,30361.160020,30153.808867,31619.881622,30393.934843,30181.359834


In [106]:
alias_modelos = [m.alias for m in modelos]
metricas_sarimas = metricas_cv(cv_rolling, alias_modelos)
metricas_sarimas


,Modelo,RMSE,MAPE,MAE,RMSE_STD,MAPE_STD,MAE_STD
1,"ARIMA(0,1,0)(0,1,0)_24",2091.422201,6.749839,1820.563889,1374.068642,4.851924,1286.537781
2,"ARIMA(0,1,0)(0,1,1)_24",1925.396495,6.328501,1700.595247,1347.869283,4.673525,1242.953841
3,"ARIMA(0,1,0)(0,1,2)_24",1889.534487,6.208080,1665.024987,1193.484857,4.262844,1111.316186
4,"ARIMA(0,1,1)(0,1,0)_24",2152.608697,6.959082,1873.677178,1381.538191,4.928720,1298.509292
5,"ARIMA(0,1,1)(0,1,1)_24",1957.726535,6.424324,1722.292461,1283.944181,4.490706,1182.637625
6,"ARIMA(0,1,1)(0,1,2)_24",1915.132756,6.283781,1684.154026,1155.430045,4.159934,1074.507881
7,"ARIMA(0,1,2)(0,1,0)_24",2186.737252,7.074047,1902.826785,1361.012986,4.864960,1278.741715
8,"ARIMA(0,1,2)(0,1,1)_24",2006.195313,6.591871,1765.689106,1272.844303,4.458016,1173.269048
9,"ARIMA(0,1,2)(0,1,2)_24",1975.480534,6.482986,1736.614787,1168.081422,4.208751,1091.082445
10,"ARIMA(1,1,0)(0,1,0)_24",2199.339139,7.133527,1917.518868,1377.555885,4.938723,1294.008429


In [108]:
metricas_sarimas.sort_values(by = 'RMSE', ascending = True)

,Modelo,RMSE,MAPE,MAE,RMSE_STD,MAPE_STD,MAE_STD
3,"ARIMA(0,1,0)(0,1,2)_24",1889.534487,6.208080,1665.024987,1193.484857,4.262844,1111.316186
6,"ARIMA(0,1,1)(0,1,2)_24",1915.132756,6.283781,1684.154026,1155.430045,4.159934,1074.507881
24,"ARIMA(2,1,1)(0,1,2)_24",1915.261215,6.226348,1681.923373,1094.831373,3.784383,1018.617326
2,"ARIMA(0,1,0)(0,1,1)_24",1925.396495,6.328501,1700.595247,1347.869283,4.673525,1242.953841
23,"ARIMA(2,1,1)(0,1,1)_24",1943.107030,6.306466,1707.269546,1200.911089,4.001722,1100.530878
5,"ARIMA(0,1,1)(0,1,1)_24",1957.726535,6.424324,1722.292461,1283.944181,4.490706,1182.637625
12,"ARIMA(1,1,0)(0,1,2)_24",1973.873641,6.501856,1736.519792,1172.993267,4.331850,1095.856570
9,"ARIMA(0,1,2)(0,1,2)_24",1975.480534,6.482986,1736.614787,1168.081422,4.208751,1091.082445
8,"ARIMA(0,1,2)(0,1,1)_24",2006.195313,6.591871,1765.689106,1272.844303,4.458016,1173.269048
11,"ARIMA(1,1,0)(0,1,1)_24",2024.435426,6.682198,1785.025669,1292.382951,4.625506,1192.027999
